# EDA + decisiones de limpieza y modelado
Notebook con trazabilidad: perfil inicial, criterios de imputacion/eliminacion, enriquecimiento ubigeo y propuesta de targets.

## 1) Librerias
Carga librerias para limpieza, EDA, enriquecimiento geografico y exportacion.

In [1]:
# !pip install pandas pyarrow seaborn matplotlib numpy scikit-learn
import re
import unicodedata
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 180)
sns.set(style="whitegrid")

## 2) Carga y normalizacion de nombres
Lee el CSV fuente en texto y normaliza nombres de columnas: minusculas, sin acentos, sin espacios y con `_`.

In [ ]:
CSV_PATH = "BD_2020-2025.csv"
df_raw = pd.read_csv(CSV_PATH, dtype=str, low_memory=False)

def normalizar_nombre(s: str) -> str:
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

original_cols = df_raw.columns.tolist()
new_cols = [normalizar_nombre(c) for c in original_cols]

seen = {}
final_cols = []
for c in new_cols:
    if c not in seen:
        seen[c] = 0
        final_cols.append(c)
    else:
        seen[c] += 1
        final_cols.append(f"{c}_{seen[c]}")

df = df_raw.copy()
df.columns = final_cols

if "nivel_de_riesgo_victima" in df.columns:
    df = df.rename(columns={"nivel_de_riesgo_victima": "nivel_riesgo_victima"})
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]:,}")
pd.DataFrame({"original": original_cols[:20], "normalizada": final_cols[:20]})

Filas: 936,835 | Columnas: 186


,original,normalizada
0,CEM,cem
1,CONDICION,condicion
2,FECHA_INGRESO,fecha_ingreso
3,INFORMANTE,informante
4,FORMA_INGRESO,forma_ingreso
5,VICTIMA_PERUANA,victima_peruana
6,VICTIMA_CUENTA_DNI,victima_cuenta_dni
7,VICTIMA_EXTRANJERA,victima_extranjera
8,VICTIMA_PAIS_EXTRANJERO,victima_pais_extranjero
9,VICTIMA_CARNE_EXTRANJERIA,victima_carne_extranjeria


## 3) Limpieza base
Limpia espacios y estandariza vacios a `NA` para medir nulos y preparar cruces correctamente.

In [3]:
for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

df = df.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA, "NaN": pd.NA})
dup_exactos = df.duplicated().sum()
print(f"Duplicados exactos: {dup_exactos:,}")

Duplicados exactos: 2,012


## 4) Enriquecimiento UBIGEO (codigo y nombre)
Construye `ubigeo_codigo` uniendo `dpto_domicilio + prov_domicilio + dist_domicilio` con padding a 2 digitos por nivel.
Luego cruza contra el maestro `/home/munasqa/MAESTRIA/opencode/ubigeo_trabajar.csv` para crear `ubigeo_nombre`.

In [ ]:
UBIGEO_PATH = "ubigeo_trabajar.csv"
ub_raw = pd.read_csv(UBIGEO_PATH, dtype=str, low_memory=False)
ub_raw.columns = [normalizar_nombre(c) for c in ub_raw.columns]

for c in ["codigo_dpto", "codigo_prov", "codigo_dist"]:
    if c in ub_raw.columns:
        ub_raw[c] = ub_raw[c].astype(str).str.strip().str.zfill(2)

ub_raw["ubigeo_codigo"] = ub_raw["codigo_dpto"] + ub_raw["codigo_prov"] + ub_raw["codigo_dist"]
ub_raw["ubigeo_nombre"] = ub_raw["dpto"].astype(str).str.strip() + " | " + ub_raw["prov"].astype(str).str.strip() + " | " + ub_raw["dist"].astype(str).str.strip()
ub_map = ub_raw[["ubigeo_codigo", "ubigeo_nombre"]].drop_duplicates()

required_cols = ["dpto_domicilio", "prov_domicilio", "dist_domicilio"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"No se encontraron columnas requeridas para ubigeo: {missing_required}")

for c in required_cols:
    df[c] = df[c].astype(str).str.strip().replace({"<NA>": pd.NA}).str.zfill(2)

# evitar crear codigo si alguna parte falta
mask_partes = df[required_cols].notna().all(axis=1)
df["ubigeo_codigo"] = pd.NA
df.loc[mask_partes, "ubigeo_codigo"] = (
    df.loc[mask_partes, "dpto_domicilio"] +
    df.loc[mask_partes, "prov_domicilio"] +
    df.loc[mask_partes, "dist_domicilio"]
)

df = df.merge(ub_map, on="ubigeo_codigo", how="left")

match_rate = df["ubigeo_nombre"].notna().mean() * 100
print(f"Coincidencia ubigeo_nombre: {match_rate:.2f}%")
display(df[["dpto_domicilio", "prov_domicilio", "dist_domicilio", "ubigeo_codigo", "ubigeo_nombre"]].head(10))

Coincidencia ubigeo_nombre: 99.99%


,dpto_domicilio,prov_domicilio,dist_domicilio,ubigeo_codigo,ubigeo_nombre
0,03,01,01,030101,Apurímac | Abancay | Abancay
1,03,01,01,030101,Apurímac | Abancay | Abancay
2,03,01,01,030101,Apurímac | Abancay | Abancay
3,04,01,22,040122,Arequipa | Arequipa | Socabaya
4,03,01,06,030106,Apurímac | Abancay | Lambrama
5,03,01,01,030101,Apurímac | Abancay | Abancay
6,03,01,01,030101,Apurímac | Abancay | Abancay
7,03,01,01,030101,Apurímac | Abancay | Abancay
8,03,01,01,030101,Apurímac | Abancay | Abancay
9,03,01,01,030101,Apurímac | Abancay | Abancay


## 5) Tabla markdown inicial (variables, unicos, % nulos)
Tabla de perfil por variable para justificar decisiones de imputacion/eliminacion.

In [5]:
perfil_inicial = pd.DataFrame({
    "variable": df.columns,
    "dtype_inicial": [str(df[c].dtype) for c in df.columns],
    "n_unicos": [int(df[c].nunique(dropna=True)) for c in df.columns],
    "nulos": [int(df[c].isna().sum()) for c in df.columns],
    "pct_nulos": [round(df[c].isna().mean() * 100, 4) for c in df.columns]
}).sort_values(["pct_nulos", "n_unicos"], ascending=[False, False])

display(perfil_inicial.head(30))
md_inicial = perfil_inicial.to_markdown(index=False)
print(md_inicial[:6000])

,variable,dtype_inicial,n_unicos,nulos,pct_nulos
146,violencia_carcelaria,object,1,936834,99.9999
14,victima_apatrida,object,1,936830,99.9995
49,agresor_apatrida,object,1,936828,99.9993
48,agresor_asilado,object,1,936824,99.9988
13,victima_asilado,object,1,936820,99.9984
47,agresor_solicitante_asilo,object,1,936815,99.9979
12,victima_solicitante_asilo,object,1,936803,99.9966
174,acoso_politico,object,1,936796,99.9958
69,percepcion_salario_menor,object,1,936748,99.9907
175,violencia_tics,object,1,936716,99.9873


| variable                                | dtype_inicial   |   n_unicos |   nulos |   pct_nulos |
|:----------------------------------------|:----------------|-----------:|--------:|------------:|
| violencia_carcelaria                    | object          |          1 |  936834 |     99.9999 |
| victima_apatrida                        | object          |          1 |  936830 |     99.9995 |
| agresor_apatrida                        | object          |          1 |  936828 |     99.9993 |
| agresor_asilado                         | object          |          1 |  936824 |     99.9988 |
| victima_asilado                         | object          |          1 |  936820 |     99.9984 |
| agresor_solicitante_asilo               | object          |          1 |  936815 |     99.9979 |
| victima_solicitante_asilo               | object          |          1 |  936803 |     99.9966 |
| acoso_politico                          | object          |          1 |  936796 |     99.9958 |
| percepci

## 6) Tipado analitico
Convierte fecha y numericas clave para analisis temporal/estadistico.

In [6]:
if "fecha_ingreso" in df.columns:
    df["fecha_ingreso"] = pd.to_datetime(df["fecha_ingreso"], errors="coerce")
    df["anio"] = df["fecha_ingreso"].dt.year
    df["mes"] = df["fecha_ingreso"].dt.month
    df["anio_mes"] = df["fecha_ingreso"].dt.to_period("M").astype(str)

for c in ["edad_victima", "edad_agresor", "hijas_vivas", "hijos_vivos"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

## 7) Tipado semantico (clave metodologica)
Como la mayoria de variables vienen codificadas numericamente, se define un tipo semantico por columna.
- `numerica_real`: magnitud (edad, conteos).
- `categorica_codificada`: codigos de respuesta (aunque sean numeros).
- `fecha`: variable de fecha.
- `temporal_derivada`: anio/mes derivados de fecha.

In [7]:
columnas_numericas_reales = ["edad_victima", "edad_agresor", "hijas_vivas", "hijos_vivos"]

tipo_semantico = {}
for c in df.columns:
    if c == "fecha_ingreso":
        tipo_semantico[c] = "fecha"
    elif c in ["anio", "mes"]:
        tipo_semantico[c] = "temporal_derivada"
    elif c in columnas_numericas_reales:
        tipo_semantico[c] = "numerica_real"
    else:
        tipo_semantico[c] = "categorica_codificada"

tabla_tipos = pd.DataFrame({
    "variable": df.columns,
    "tipo_semantico": [tipo_semantico[c] for c in df.columns],
    "n_unicos": [int(df[c].nunique(dropna=True)) for c in df.columns],
    "pct_nulos": [round(df[c].isna().mean() * 100, 4) for c in df.columns]
}).sort_values(["tipo_semantico", "n_unicos"], ascending=[True, False])

display(tabla_tipos.head(40))

,variable,tipo_semantico,n_unicos,pct_nulos
186,ubigeo_codigo,categorica_codificada,1883,0.0000
187,ubigeo_nombre,categorica_codificada,1882,0.0106
0,cem,categorica_codificada,449,0.0000
27,centro_poblado_domicilio,categorica_codificada,362,0.0000
35,ocupacion_victima,categorica_codificada,113,66.8244
61,ocupacion_agresor,categorica_codificada,113,27.2448
43,agresor_pais_extranjero,categorica_codificada,84,98.5646
8,victima_pais_extranjero,categorica_codificada,79,98.3060
190,anio_mes,categorica_codificada,69,0.0000
112,n_anos,categorica_codificada,62,82.9198


## 8) Reglas de eliminacion (explicitas)
Elimina columnas con `% nulos > 99` y conserva el resto para imputacion/modelado.

In [8]:
THRESH_NULL = 99.0
cols_drop_null = perfil_inicial.loc[perfil_inicial["pct_nulos"] > THRESH_NULL, "variable"].tolist()

decision_null = perfil_inicial[["variable", "pct_nulos"]].copy()
decision_null["decision"] = np.where(decision_null["variable"].isin(cols_drop_null),
                                    f"eliminar (pct_nulos > {THRESH_NULL}%)",
                                    "conservar")

df_model = df.drop(columns=cols_drop_null).copy()
print(f"Eliminadas por nulos: {len(cols_drop_null)}")
print(f"Shape actual: {df_model.shape}")
display(decision_null[decision_null['decision'] != 'conservar'].head(20))

Eliminadas por nulos: 57
Shape actual: (936835, 134)


,variable,pct_nulos,decision
146,violencia_carcelaria,99.9999,eliminar (pct_nulos > 99.0%)
14,victima_apatrida,99.9995,eliminar (pct_nulos > 99.0%)
49,agresor_apatrida,99.9993,eliminar (pct_nulos > 99.0%)
48,agresor_asilado,99.9988,eliminar (pct_nulos > 99.0%)
13,victima_asilado,99.9984,eliminar (pct_nulos > 99.0%)
47,agresor_solicitante_asilo,99.9979,eliminar (pct_nulos > 99.0%)
12,victima_solicitante_asilo,99.9966,eliminar (pct_nulos > 99.0%)
174,acoso_politico,99.9958,eliminar (pct_nulos > 99.0%)
69,percepcion_salario_menor,99.9907,eliminar (pct_nulos > 99.0%)
175,violencia_tics,99.9873,eliminar (pct_nulos > 99.0%)


## 9) Imputacion (explicita y trazable por tipo semantico)
Imputa para dejar base sin nulos con reglas semanticas:
- `numerica_real` y `temporal_derivada`: mediana,
- `categorica_codificada`: `desconocido`,
- `fecha`: `1900-01-01`.

In [9]:
df_imp = df_model.copy()
missing_before = df_imp.isna().sum()
imputacion = {}

for c in df_imp.columns:
    n_null = int(df_imp[c].isna().sum())
    if n_null == 0:
        imputacion[c] = "sin imputacion (sin nulos)"
        continue

    if tipo_semantico.get(c) in ["numerica_real", "temporal_derivada"]:
        med = df_imp[c].median()
        df_imp[c] = df_imp[c].fillna(med)
        imputacion[c] = f"mediana ({med})"
    elif tipo_semantico.get(c) == "fecha":
        df_imp[c] = df_imp[c].fillna(pd.Timestamp("1900-01-01"))
        imputacion[c] = "fecha fija (1900-01-01)"
    else:
        df_imp[c] = df_imp[c].fillna("desconocido")
        imputacion[c] = "categoria fija (desconocido)"

missing_after = df_imp.isna().sum()
print(f"Nulos antes: {int(missing_before.sum()):,}")
print(f"Nulos despues: {int(missing_after.sum()):,}")

Nulos antes: 61,743,194
Nulos despues: 0


## 10) Baja varianza (variables casi constantes)
Evalua variables casi constantes con enfoque de bajo consumo de RAM.
Se evita construir una matriz numerica grande y se usa proporcion de la moda por variable.

In [10]:
# 1) Reducir memoria antes del calculo
for c in df_imp.select_dtypes(include=["float64"]).columns:
    df_imp[c] = pd.to_numeric(df_imp[c], downcast="float")

for c in df_imp.select_dtypes(include=["int64"]).columns:
    df_imp[c] = pd.to_numeric(df_imp[c], downcast="integer")

for c in df_imp.select_dtypes(include=["object"]).columns:
    nun = df_imp[c].nunique(dropna=False)
    if nun <= 100:
        df_imp[c] = df_imp[c].astype("category")

# 2) Baja varianza por proporcion de la moda (sin matriz grande)
LOW_INFO_THRESH = 0.995
bin_candidates = []
low_var_cols = []

for c in df_imp.columns:
    if tipo_semantico.get(c) not in ["categorica_codificada", "numerica_real", "temporal_derivada"]:
        continue

    vc = df_imp[c].value_counts(dropna=False)
    if len(vc) == 0:
        continue

    if len(vc) <= 2:
        bin_candidates.append(c)
        p_top = vc.iloc[0] / len(df_imp[c])
        if p_top >= LOW_INFO_THRESH:
            low_var_cols.append(c)

print(f"Binarias evaluadas: {len(bin_candidates)}")
print(f"Eliminables por baja varianza (moda >= {LOW_INFO_THRESH*100:.2f}%): {len(low_var_cols)}")

df_final = df_imp.drop(columns=low_var_cols, errors="ignore").copy()
print(f"Shape final: {df_final.shape}")

Binarias evaluadas: 56
Eliminables por baja varianza (moda >= 99.50%): 1
Shape final: (936835, 133)


## 11) Exclusión de violencia economica (decision EDA)
Por decision analitica, se excluyen registros y variables asociadas a violencia economica porque su muestra no es representativa para los modelos objetivo en esta etapa.
- Registros: se eliminan filas con `tipo_violencia == 0` (economica).
- Variables: se eliminan columnas vinculadas a violencia economica/patrimonial cuando existan.
Esta decision queda documentada para diccionario y trazabilidad.

In [11]:
filas_antes_economica = len(df_final)

# 1) eliminar registros de tipo_violencia economica (0)
if "tipo_violencia" in df_final.columns:
    tv_num = pd.to_numeric(df_final["tipo_violencia"], errors="coerce")
    df_final = df_final[tv_num != 0].copy()

# 2) eliminar variables economicas/patrimoniales (si existen)
patrones_economica = [
    "patrim", "econom", "ingreso", "salario", "alimentaria",
    "posesion", "tenencia", "recursos", "medios_indispensables"
]
econ_cols = [c for c in df_final.columns if any(p in c for p in patrones_economica)]

# no eliminar la target principal en caso de match por nombre
econ_cols = [c for c in econ_cols if c != "tipo_violencia"]
df_final = df_final.drop(columns=econ_cols, errors="ignore")

filas_despues_economica = len(df_final)
print(f"Filas antes exclusion economica: {filas_antes_economica:,}")
print(f"Filas despues exclusion economica: {filas_despues_economica:,}")
print(f"Filas eliminadas: {filas_antes_economica - filas_despues_economica:,}")
print(f"Columnas economicas eliminadas: {len(econ_cols)}")

Filas antes exclusion economica: 936,835
Filas despues exclusion economica: 932,860
Filas eliminadas: 3,975
Columnas economicas eliminadas: 4


## 12) Recodificacion final de targets a base 0
Para modelado (por ejemplo XGBoost multiclase), se recodifican los targets a clases consecutivas desde 0.
- `tipo_violencia`: 1,2,3 -> 0,1,2 (psicologica,fisica,sexual).
- `nivel_riesgo_victima`: 1,2,3 -> 0,1,2 (bajo,medio,alto).
Se guardan columnas `*_orig` para trazabilidad.

In [ ]:
# respaldo para trazabilidad
if "tipo_violencia" in df_final.columns:
    df_final["tipo_violencia_orig"] = pd.to_numeric(df_final["tipo_violencia"], errors="coerce")
    df_final["tipo_violencia"] = df_final["tipo_violencia_orig"].map({1: 0, 2: 1, 3: 2})

if "nivel_riesgo_victima" in df_final.columns:
    df_final["nivel_riesgo_victima_orig"] = pd.to_numeric(df_final["nivel_riesgo_victima"], errors="coerce")
    df_final["nivel_riesgo_victima"] = df_final["nivel_riesgo_victima_orig"] - 1

print("Categorias finales tipo_violencia:", sorted(df_final["tipo_violencia"].dropna().unique().tolist()) if "tipo_violencia" in df_final.columns else "NA")
print("Categorias finales nivel_riesgo_victima:", sorted(df_final["nivel_riesgo_victima"].dropna().unique().tolist()) if "nivel_riesgo_victima" in df_final.columns else "NA")


## 12) Tabla markdown final (trazabilidad completa)
Incluye variables finales con tipo, cardinalidad, nulos antes/despues e imputacion aplicada.

In [12]:
reporte = pd.DataFrame({
    "variable": df_model.columns,
    "tipo_semantico": [tipo_semantico.get(c, "no_definido") for c in df_model.columns],
    "dtype_post_tipado": [str(df_model[c].dtype) for c in df_model.columns],
    "n_unicos": [int(df_model[c].nunique(dropna=True)) for c in df_model.columns],
    "nulos_antes_imputacion": [int(missing_before[c]) for c in df_model.columns],
    "imputacion": [imputacion[c] for c in df_model.columns],
    "nulos_despues_imputacion": [int(missing_after[c]) for c in df_model.columns]
})

reporte["decision_final"] = "conservar"
reporte.loc[reporte["variable"].isin(low_var_cols), "decision_final"] = "eliminar (baja varianza)"
reporte.loc[reporte["variable"].isin(econ_cols), "decision_final"] = "eliminar (relacionada a violencia economica)"

reporte = reporte.sort_values(["decision_final", "nulos_antes_imputacion"], ascending=[True, False])
display(reporte.head(40))

reporte_final = reporte[~reporte['variable'].isin(low_var_cols + econ_cols)].copy()
md_final = reporte_final.to_markdown(index=False)
print(md_final[:6000])

,variable,tipo_semantico,dtype_post_tipado,n_unicos,nulos_antes_imputacion,imputacion,nulos_despues_imputacion,decision_final
97,vinculo_afectivo_otro,categorica_codificada,object,1,926697,categoria fija (desconocido),0,conservar
31,otro_seguro,categorica_codificada,object,3,925465,categoria fija (desconocido),0,conservar
63,abandono,categorica_codificada,object,1,924972,categoria fija (desconocido),0,conservar
106,factor_victima_discapacidad,categorica_codificada,object,1,924590,categoria fija (desconocido),0,conservar
79,otra_vsex,categorica_codificada,object,1,924526,categoria fija (desconocido),0,conservar
57,prohibe_recibir_visitas,categorica_codificada,object,1,923741,categoria fija (desconocido),0,conservar
74,golpes_con_objetos_contundentes,categorica_codificada,object,1,923613,categoria fija (desconocido),0,conservar
36,agresor_pais_extranjero,categorica_codificada,object,84,923388,categoria fija (desconocido),0,conservar
37,agresor_carne_extranjeria,categorica_codificada,object,2,923253,categoria fija (desconocido),0,conservar
58,prohibe_estudiar_trabajar_salir,categorica_codificada,object,1,923205,categoria fija (desconocido),0,conservar


| variable                          | tipo_semantico        | dtype_post_tipado   |   n_unicos |   nulos_antes_imputacion | imputacion                   |   nulos_despues_imputacion | decision_final   |
|:----------------------------------|:----------------------|:--------------------|-----------:|-------------------------:|:-----------------------------|---------------------------:|:-----------------|
| vinculo_afectivo_otro             | categorica_codificada | object              |          1 |                   926697 | categoria fija (desconocido) |                          0 | conservar        |
| otro_seguro                       | categorica_codificada | object              |          3 |                   925465 | categoria fija (desconocido) |                          0 | conservar        |
| abandono                          | categorica_codificada | object              |          1 |                   924972 | categoria fija (desconocido) |                          0 | cons

## 13) Targets definidos para modelado
Se usan como targets finales `tipo_violencia` y `nivel_riesgo_victima`. Esta celda valida su disponibilidad y muestra su distribucion porcentual.

In [13]:
targets_definidos = ["tipo_violencia", "nivel_riesgo_victima"]
targets_disponibles = [c for c in targets_definidos if c in df_final.columns]
targets_faltantes = [c for c in targets_definidos if c not in df_final.columns]

print("Targets disponibles:", targets_disponibles)
print("Targets faltantes:", targets_faltantes)

for t in targets_disponibles:
    print("\nDistribucion % de", t)
    print(df_final[t].value_counts(normalize=True, dropna=False).rename("proporcion").mul(100).round(2))


Targets disponibles: ['tipo_violencia', 'nivel_riesgo_victima']
Targets faltantes: []

Distribucion % de tipo_violencia
tipo_violencia
1    44.52
2    38.43
3    17.05
0     0.00
Name: proporcion, dtype: float64

Distribucion % de nivel_riesgo_victima
nivel_riesgo_victima
2    51.87
3    26.74
1    21.39
Name: proporcion, dtype: float64


## 14) Exportables
Guarda reportes markdown/csv y exporta la base final en parquet (unico y particionado por anio).

In [14]:
OUT_MD_INI = "/home/munasqa/MAESTRIA/opencode/reporte_variables_inicial.md"
OUT_CSV_INI = "/home/munasqa/MAESTRIA/opencode/reporte_variables_inicial.csv"
OUT_MD_FIN = "/home/munasqa/MAESTRIA/opencode/reporte_variables_final.md"
OUT_CSV_FIN = "/home/munasqa/MAESTRIA/opencode/reporte_variables_final.csv"

perfil_inicial.to_csv(OUT_CSV_INI, index=False, encoding="utf-8")
with open(OUT_MD_INI, "w", encoding="utf-8") as f:
    f.write(md_inicial)

reporte_final.to_csv(OUT_CSV_FIN, index=False, encoding="utf-8")
with open(OUT_MD_FIN, "w", encoding="utf-8") as f:
    f.write(md_final)

OUT_PARQUET = "/home/munasqa/MAESTRIA/opencode/base_modelado.parquet"
df_final.to_parquet(OUT_PARQUET, index=False, engine="pyarrow")

OUT_PART = "/home/munasqa/MAESTRIA/opencode/base_modelado_particionado"
if "anio" in df_final.columns:
    dfp = df_final[df_final["anio"].notna()].copy()
    dfp["anio"] = dfp["anio"].astype(int)
    dfp.to_parquet(OUT_PART, index=False, engine="pyarrow", partition_cols=["anio"])

print("Listo.")
print(OUT_MD_INI)
print(OUT_CSV_INI)
print(OUT_MD_FIN)
print(OUT_CSV_FIN)
print(OUT_PARQUET)
print(OUT_PART)

Listo.
/home/munasqa/MAESTRIA/opencode/reporte_variables_inicial.md
/home/munasqa/MAESTRIA/opencode/reporte_variables_inicial.csv
/home/munasqa/MAESTRIA/opencode/reporte_variables_final.md
/home/munasqa/MAESTRIA/opencode/reporte_variables_final.csv
/home/munasqa/MAESTRIA/opencode/base_modelado.parquet
/home/munasqa/MAESTRIA/opencode/base_modelado_particionado
